# Day 079 | Graph Neural Networks Basics

## 100 Days of ML for Geotechnical Engineering
**Phase:** Deep Learning | **Author:** Ripon Chandra Malo | **Date:** YYYY-MM-DD

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Graph representation | Nodes + edges encode structure | Soil particles as nodes, contacts as edges |
| Adjacency matrix | Matrix encoding connections | Which particles/elements are neighbors |
| Node features | Properties at each node | Particle size, stress, displacement |
| Message passing | Aggregate neighbor information | Each particle collects forces from neighbors |
| Graph convolution | Learnable neighbor aggregation | Weighted combination of neighbor features |
| FEM mesh as graph | Elements and nodes form a graph | Triangular mesh of a slope or foundation |

---

## Our Geotechnical Problem

We will represent a **soil particle network** and a **FEM mesh** as graphs, then implement
**message passing** to predict node-level properties (e.g., displacement or stress).

```
  Soil Particle Network          FEM Mesh (Foundation)
  o--o--o                        ^--^--^--^
  |\ |/ |                        |\ |\ |\ |
  o--o--o      -------->         ^--^--^--^
  |/ |\ |     Graph NN           |\ |\ |\ |
  o--o--o                        ^--^--^--^
  
  Nodes = particles/elements    Predict displacement at each node
  Edges = contacts/connections
```

## Part 1: Import Libraries and Graph Basics

In [ ]:
# ───────────────────────────────────────────────────────
# IMPORTS AND CONFIGURATION
# ───────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

print("Graph Neural Networks for Geotechnical Engineering")
print("=" * 50)
print("\nGraphs are natural representations for:")
print("  - Soil particle assemblies (DEM)")
print("  - FEM meshes")
print("  - Sensor networks on structures")
print("  - Geological layer connectivity")

---

## Part 2: Create a Soil Particle Graph

We create a simple 2D particle assembly and connect particles that are within a contact distance.

In [ ]:
# ───────────────────────────────────────────────────────
# CREATE A 2D SOIL PARTICLE ASSEMBLY GRAPH
# ───────────────────────────────────────────────────────

def create_particle_graph(n_particles=20, box_size=10.0, contact_radius=3.5):
    """
    Create a graph from a 2D particle assembly.
    Particles within contact_radius are connected.
    """
    # Random particle positions
    positions = np.random.rand(n_particles, 2) * box_size
    
    # Particle properties (node features)
    radii = np.random.uniform(0.3, 0.8, n_particles)          # particle radius
    densities = np.random.uniform(2.5, 2.8, n_particles)      # g/cm3
    
    # Build adjacency: connect particles within contact distance
    edge_src, edge_dst = [], []
    for i in range(n_particles):
        for j in range(i+1, n_particles):
            dist = np.linalg.norm(positions[i] - positions[j])
            if dist < contact_radius:
                edge_src.extend([i, j])  # undirected: both ways
                edge_dst.extend([j, i])
    
    edge_index = np.array([edge_src, edge_dst])  # (2, n_edges)
    
    # Node features: [x_pos, y_pos, radius, density]
    node_features = np.column_stack([positions, radii, densities])
    
    return positions, node_features, edge_index

positions, node_features, edge_index = create_particle_graph(n_particles=25)

n_nodes = node_features.shape[0]
n_edges = edge_index.shape[1]

print(f"Particle assembly graph:")
print(f"  Nodes (particles):  {n_nodes}")
print(f"  Edges (contacts):   {n_edges}")
print(f"  Avg connections:    {n_edges / n_nodes:.1f}")
print(f"  Node feature dim:   {node_features.shape[1]} (x, y, radius, density)")

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Draw edges
for i in range(0, n_edges, 2):  # every other (undirected)
    src, dst = edge_index[0, i], edge_index[1, i]
    ax.plot([positions[src, 0], positions[dst, 0]],
            [positions[src, 1], positions[dst, 1]],
            'k-', alpha=0.3, linewidth=0.8)

# Draw nodes (sized by particle radius)
sizes = node_features[:, 2] * 300
scatter = ax.scatter(positions[:, 0], positions[:, 1], s=sizes,
                     c=node_features[:, 3], cmap='YlOrBr',
                     edgecolors='k', linewidth=1, zorder=5)
plt.colorbar(scatter, ax=ax, label='Density (g/cm3)')

for i in range(n_nodes):
    ax.annotate(str(i), (positions[i, 0], positions[i, 1]),
                ha='center', va='center', fontsize=7, fontweight='bold')

ax.set_xlabel('x (m)', fontsize=12)
ax.set_ylabel('y (m)', fontsize=12)
ax.set_title('Soil Particle Assembly as a Graph', fontsize=13)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Part 3: Adjacency Matrix and Degree Matrix

The **adjacency matrix** A encodes the graph structure:
- A[i,j] = 1 if node i connects to node j
- The **degree matrix** D has the count of neighbors on the diagonal

In [ ]:
# ───────────────────────────────────────────────────────
# ADJACENCY AND DEGREE MATRICES
# ───────────────────────────────────────────────────────

def build_adjacency_matrix(edge_index, n_nodes):
    """Build dense adjacency matrix from edge index."""
    A = np.zeros((n_nodes, n_nodes))
    for i in range(edge_index.shape[1]):
        A[edge_index[0, i], edge_index[1, i]] = 1.0
    return A

A = build_adjacency_matrix(edge_index, n_nodes)
D = np.diag(A.sum(axis=1))  # degree matrix

print(f"Adjacency matrix shape: {A.shape}")
print(f"Degree matrix diagonal: {np.diag(D).astype(int)}")
print(f"\nMin degree: {int(np.diag(D).min())} connections")
print(f"Max degree: {int(np.diag(D).max())} connections")

# Visualize adjacency matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(A, cmap='Blues', interpolation='nearest')
axes[0].set_title('Adjacency Matrix A', fontsize=13)
axes[0].set_xlabel('Node j')
axes[0].set_ylabel('Node i')

axes[1].bar(range(n_nodes), np.diag(D), color='steelblue', edgecolor='k')
axes[1].set_xlabel('Node index', fontsize=12)
axes[1].set_ylabel('Degree (# neighbors)', fontsize=12)
axes[1].set_title('Node Degree Distribution', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nThe adjacency matrix is symmetric (undirected graph).")
print(f"Symmetry check: {np.allclose(A, A.T)}")

---

## Part 4: Message Passing — The Core of GNNs

In message passing, each node **aggregates features from its neighbors**:

$$h_i^{(l+1)} = \sigma\left( W \cdot \text{AGGREGATE}\left(\{h_j^{(l)} : j \in \mathcal{N}(i)\}\right) \right)$$

This is analogous to a particle summing forces from all its contact neighbors.

In [ ]:
# ───────────────────────────────────────────────────────
# MANUAL MESSAGE PASSING STEP
# ───────────────────────────────────────────────────────

def message_passing_step(node_features, adjacency, W, b):
    """
    One step of message passing (graph convolution).
    
    h_new = ReLU( D_hat_inv @ A_hat @ h @ W + b )
    
    D_hat_inv @ A_hat normalizes by degree (mean aggregation).
    """
    # Add self-loops: each node also sees itself
    A_hat = adjacency + np.eye(adjacency.shape[0])
    D_hat_diag = A_hat.sum(axis=1)
    D_hat_inv = np.diag(1.0 / D_hat_diag)
    
    # Aggregate neighbor features (with self-loop)
    aggregated = D_hat_inv @ A_hat @ node_features  # (N, F_in)
    
    # Linear transformation + activation
    output = aggregated @ W + b   # (N, F_out)
    output = np.maximum(0, output)  # ReLU
    
    return output

# Initialize random weights
F_in = node_features.shape[1]   # 4 input features
F_out = 8                        # 8 output features

W = np.random.randn(F_in, F_out) * 0.5
b = np.zeros(F_out)

# Run one message passing step
h_new = message_passing_step(node_features, A, W, b)

print(f"Input features:  {node_features.shape}  (N, F_in={F_in})")
print(f"Output features: {h_new.shape}  (N, F_out={F_out})")
print(f"\nEach node now has features that incorporate neighbor information.")
print(f"\nNode 0 original features: {node_features[0]}")
print(f"Node 0 after message passing: {h_new[0]}")

---

## Part 5: Build a GNN in PyTorch

Now we implement a proper Graph Neural Network using PyTorch for learning on a small geotechnical mesh.

In [ ]:
# ───────────────────────────────────────────────────────
# GNN MODEL IN PYTORCH
# ───────────────────────────────────────────────────────

class GraphConvLayer(nn.Module):
    """Simple graph convolution layer (GCN-style)."""
    
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
    
    def forward(self, x, adj):
        """
        Args:
            x:   node features (N, F_in)
            adj: adjacency matrix with self-loops (N, N)
        """
        # Degree normalization: D^{-1/2} A D^{-1/2}
        deg = adj.sum(dim=1)  # (N,)
        deg_inv_sqrt = torch.pow(deg, -0.5)
        deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
        D_inv_sqrt = torch.diag(deg_inv_sqrt)
        
        # Symmetric normalization
        adj_norm = D_inv_sqrt @ adj @ D_inv_sqrt
        
        # Message passing + linear transform
        support = self.linear(x)       # (N, F_out)
        output = adj_norm @ support    # (N, F_out)
        
        return output


class GeoGNN(nn.Module):
    """GNN for geotechnical node-level prediction."""
    
    def __init__(self, in_features, hidden_dim=32, out_features=1):
        super().__init__()
        self.conv1 = GraphConvLayer(in_features, hidden_dim)
        self.conv2 = GraphConvLayer(hidden_dim, hidden_dim)
        self.conv3 = GraphConvLayer(hidden_dim, out_features)
    
    def forward(self, x, adj):
        h = F.relu(self.conv1(x, adj))
        h = F.relu(self.conv2(h, adj))
        h = self.conv3(h, adj)
        return h

model = GeoGNN(in_features=4, hidden_dim=32, out_features=1)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---

## Part 6: Create a Geotechnical FEM Mesh Graph

We create a simple structured mesh for a strip footing problem and train the GNN to predict displacement.

In [ ]:
# ───────────────────────────────────────────────────────
# CREATE A SIMPLE FEM MESH FOR STRIP FOOTING
# ───────────────────────────────────────────────────────

def create_fem_mesh(nx=8, ny=6, width=10.0, depth=6.0):
    """
    Create a structured mesh and convert to graph.
    Simulates soil under a strip footing.
    """
    # Node positions
    x = np.linspace(0, width, nx)
    y = np.linspace(0, depth, ny)
    X, Y = np.meshgrid(x, y)
    positions = np.column_stack([X.ravel(), Y.ravel()])
    n_nodes = positions.shape[0]
    
    # Node features: [x, y, depth_from_surface, distance_from_center]
    depth_from_surface = depth - positions[:, 1]
    dist_from_center = np.abs(positions[:, 0] - width / 2)
    node_features = np.column_stack([
        positions,
        depth_from_surface,
        dist_from_center
    ])
    
    # Build edges from grid connectivity
    edge_src, edge_dst = [], []
    for i in range(ny):
        for j in range(nx):
            idx = i * nx + j
            # Right neighbor
            if j < nx - 1:
                edge_src.extend([idx, idx + 1])
                edge_dst.extend([idx + 1, idx])
            # Top neighbor
            if i < ny - 1:
                edge_src.extend([idx, idx + nx])
                edge_dst.extend([idx + nx, idx])
            # Diagonal
            if j < nx - 1 and i < ny - 1:
                edge_src.extend([idx, idx + nx + 1])
                edge_dst.extend([idx + nx + 1, idx])
    
    edge_index = np.array([edge_src, edge_dst])
    
    # Synthetic target: vertical displacement (Boussinesq-inspired)
    footing_load = 100.0  # kPa
    E = 20000.0  # kPa (soil modulus)
    
    r = np.sqrt(dist_from_center**2 + depth_from_surface**2 + 0.1)
    displacement = footing_load / (np.pi * E) * (
        1.0 / np.maximum(r, 0.5)
    ) * 1000  # mm
    
    displacement = displacement * np.exp(-dist_from_center / 3.0)
    displacement += np.random.normal(0, 0.05, n_nodes)
    displacement = np.maximum(displacement, 0)
    
    return positions, node_features, edge_index, displacement

mesh_pos, mesh_features, mesh_edges, mesh_disp = create_fem_mesh()

print(f"FEM Mesh Graph:")
print(f"  Nodes: {mesh_features.shape[0]}")
print(f"  Edges: {mesh_edges.shape[1]}")
print(f"  Node features: {mesh_features.shape[1]} (x, y, depth, dist_center)")
print(f"  Displacement range: {mesh_disp.min():.3f} to {mesh_disp.max():.3f} mm")

# Visualize mesh with displacements
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mesh structure
for i in range(0, mesh_edges.shape[1], 2):
    s, d = mesh_edges[0, i], mesh_edges[1, i]
    axes[0].plot([mesh_pos[s, 0], mesh_pos[d, 0]],
                 [mesh_pos[s, 1], mesh_pos[d, 1]], 'k-', alpha=0.3, linewidth=0.5)
sc1 = axes[0].scatter(mesh_pos[:, 0], mesh_pos[:, 1], c=mesh_disp,
                      cmap='YlOrRd', s=60, edgecolors='k', linewidth=0.5, zorder=5)
plt.colorbar(sc1, ax=axes[0], label='Displacement (mm)')
axes[0].set_title('FEM Mesh with Strip Footing', fontsize=13)
axes[0].set_xlabel('x (m)')
axes[0].set_ylabel('y (m)')
axes[0].set_aspect('equal')

# Feature distribution
axes[1].hist(mesh_disp, bins=15, color='steelblue', edgecolor='k', alpha=0.7)
axes[1].set_xlabel('Displacement (mm)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Displacement Distribution', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Part 7: Train GNN on Mesh Data

In [ ]:
# ───────────────────────────────────────────────────────
# TRAIN GNN TO PREDICT NODE DISPLACEMENTS
# ───────────────────────────────────────────────────────

# Prepare PyTorch tensors
X_tensor = torch.tensor(mesh_features, dtype=torch.float32)
y_tensor = torch.tensor(mesh_disp, dtype=torch.float32).unsqueeze(1)

# Normalize features
X_mean, X_std = X_tensor.mean(dim=0), X_tensor.std(dim=0)
X_norm = (X_tensor - X_mean) / (X_std + 1e-8)

y_mean, y_std = y_tensor.mean(), y_tensor.std()
y_norm = (y_tensor - y_mean) / (y_std + 1e-8)

# Build adjacency matrix with self-loops
n_mesh_nodes = mesh_features.shape[0]
A_mesh = build_adjacency_matrix(mesh_edges, n_mesh_nodes)
A_mesh += np.eye(n_mesh_nodes)  # self-loops
A_tensor = torch.tensor(A_mesh, dtype=torch.float32)

# Initialize model
gnn = GeoGNN(in_features=4, hidden_dim=32, out_features=1)
optimizer = torch.optim.Adam(gnn.parameters(), lr=0.01)
criterion = nn.MSELoss()

# Training loop
n_epochs = 500
losses = []

for epoch in range(n_epochs):
    gnn.train()
    optimizer.zero_grad()
    
    y_pred = gnn(X_norm, A_tensor)
    loss = criterion(y_pred, y_norm)
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if epoch % 100 == 0 or epoch == n_epochs - 1:
        print(f"Epoch {epoch:4d}/{n_epochs} | Loss: {loss.item():.6f}")

print(f"\nFinal loss: {losses[-1]:.6f}")

In [ ]:
# ───────────────────────────────────────────────────────
# EVALUATE GNN PREDICTIONS
# ───────────────────────────────────────────────────────

gnn.eval()
with torch.no_grad():
    y_pred_norm = gnn(X_norm, A_tensor)
    y_pred_actual = (y_pred_norm * (y_std + 1e-8) + y_mean).numpy().flatten()

# Metrics
ss_res = np.sum((mesh_disp - y_pred_actual) ** 2)
ss_tot = np.sum((mesh_disp - mesh_disp.mean()) ** 2)
r2 = 1 - ss_res / ss_tot
rmse = np.sqrt(np.mean((mesh_disp - y_pred_actual) ** 2))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].semilogy(losses, 'b-', linewidth=1.2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('GNN Training Loss', fontsize=13)
axes[0].grid(True, alpha=0.3)

# Actual vs Predicted
axes[1].scatter(mesh_disp, y_pred_actual, alpha=0.7, edgecolors='k', linewidth=0.5, s=50)
lim = [min(mesh_disp.min(), y_pred_actual.min()),
       max(mesh_disp.max(), y_pred_actual.max())]
axes[1].plot(lim, lim, 'r--', linewidth=2)
axes[1].set_xlabel('Actual Displacement (mm)', fontsize=12)
axes[1].set_ylabel('Predicted Displacement (mm)', fontsize=12)
axes[1].set_title(f'Actual vs Predicted (R2 = {r2:.4f})', fontsize=13)
axes[1].grid(True, alpha=0.3)

# Predicted displacement on mesh
for i in range(0, mesh_edges.shape[1], 2):
    s, d = mesh_edges[0, i], mesh_edges[1, i]
    axes[2].plot([mesh_pos[s, 0], mesh_pos[d, 0]],
                 [mesh_pos[s, 1], mesh_pos[d, 1]], 'k-', alpha=0.2, linewidth=0.5)
sc = axes[2].scatter(mesh_pos[:, 0], mesh_pos[:, 1], c=y_pred_actual,
                     cmap='YlOrRd', s=60, edgecolors='k', linewidth=0.5, zorder=5)
plt.colorbar(sc, ax=axes[2], label='Predicted Disp. (mm)')
axes[2].set_title('GNN Predicted Displacements', fontsize=13)
axes[2].set_xlabel('x (m)')
axes[2].set_ylabel('y (m)')
axes[2].set_aspect('equal')

plt.tight_layout()
plt.show()

print(f"R2 = {r2:.4f} | RMSE = {rmse:.4f} mm")

---

## Part 8: Visualize Receptive Field Growth

Let us trace how information flows through the graph with each GNN layer.

In [ ]:
# ───────────────────────────────────────────────────────
# VISUALIZE RECEPTIVE FIELD GROWTH
# ───────────────────────────────────────────────────────

def get_k_hop_neighbors(adj, node_idx, k):
    """Find all nodes within k hops of a target node."""
    visited = {node_idx}
    current = {node_idx}
    hop_sets = [{node_idx}]
    
    for _ in range(k):
        next_set = set()
        for n in current:
            neighbors = set(np.where(adj[n] > 0)[0])
            next_set |= neighbors - visited
        visited |= next_set
        current = next_set
        hop_sets.append(next_set)
    
    return hop_sets

# Choose a center node
center_node = n_mesh_nodes // 2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_hop = ['red', 'orange', 'gold', 'lightblue']

for k, ax in enumerate(axes, 1):
    hop_sets = get_k_hop_neighbors(A_mesh, center_node, k)
    
    # Draw edges
    for i in range(0, mesh_edges.shape[1], 2):
        s, d = mesh_edges[0, i], mesh_edges[1, i]
        ax.plot([mesh_pos[s, 0], mesh_pos[d, 0]],
                [mesh_pos[s, 1], mesh_pos[d, 1]], 'k-', alpha=0.15, linewidth=0.5)
    
    # Color nodes by hop distance
    node_colors = ['lightgray'] * n_mesh_nodes
    for hop_idx in range(min(k + 1, len(hop_sets))):
        for n in hop_sets[hop_idx]:
            node_colors[n] = colors_hop[min(hop_idx, len(colors_hop) - 1)]
    
    ax.scatter(mesh_pos[:, 0], mesh_pos[:, 1], c=node_colors,
               s=60, edgecolors='k', linewidth=0.5, zorder=5)
    ax.scatter(mesh_pos[center_node, 0], mesh_pos[center_node, 1],
               c='red', s=120, edgecolors='k', linewidth=2, zorder=10, marker='*')
    ax.set_title(f'{k}-Hop Receptive Field ({k} GNN layers)', fontsize=12)
    ax.set_aspect('equal')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')

plt.suptitle('How GNN Layers Expand the Receptive Field', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Each GNN layer expands the receptive field by one hop.")
print("After 3 layers, the center node sees information from 3 hops away.")
print("This is how GNNs propagate information through the mesh.")

---

## Practice Exercises

### Exercise 1: Change Graph Connectivity
Increase the `contact_radius` in the particle graph. How does denser connectivity affect predictions?

### Exercise 2: Add Edge Features
Modify the message passing to include distance between nodes as an edge weight.

### Exercise 3: Deeper GNN
Add 2 more graph convolution layers. Does performance improve or degrade? (Hint: look up over-smoothing.)

In [ ]:
# EXERCISE SPACE

# Exercise 1: Try contact_radius = 2.0, 3.0, 5.0
# positions, features, edges = create_particle_graph(contact_radius=5.0)

# Exercise 2: Edge-weighted message passing
# Modify message_passing_step to weight by 1/distance

# Exercise 3: Deeper GNN — add more layers to GeoGNN
# Watch for over-smoothing: all node features become identical

---

## Key Takeaways

1. **Graphs naturally represent geotechnical structures.** Particle assemblies, FEM meshes, and sensor networks all have nodes and edges.

2. **Message passing aggregates neighbor information.** Each node updates its features by collecting and combining features from connected nodes.

3. **GNN layers expand the receptive field.** One layer sees 1-hop neighbors, two layers see 2-hop, and so on. This controls how far information travels.

4. **Adjacency normalization prevents feature explosion.** Dividing by degree ensures high-connectivity nodes do not dominate low-connectivity ones.

5. **GNNs are mesh-topology aware.** Unlike standard NNs that ignore spatial structure, GNNs respect the connectivity of the underlying problem.

---

## How This Connects to ML

| GNN Concept | Standard ML Equivalent |
|---|---|
| Node features | Feature vector per sample |
| Message passing | Convolving with neighbors (like CNN kernels) |
| Adjacency matrix | Defines which samples interact |
| Receptive field growth | CNN receptive field with deeper layers |
| Graph pooling | Global average pooling in CNNs |

**GNNs extend deep learning to irregular, non-grid data** — tomorrow we explore attention mechanisms and Transformers.

---

*Day 079 of 100 — Graphs connect everything, literally!*